# UA-DETRAC Dataset Tester & Visualizer

This notebook verifies the loading, format, and statistics of the UA-DETRAC dataset using the project's `TrafficDataset` and `get_data_loader` modules.

In [ ]:
import os
import sys
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch

# Ensure parent directory (project root) is in PYTHONPATH
project_root = os.path.dirname(os.path.dirname(os.path.abspath('')))
if project_root not in sys.path:
    sys.path.append(project_root)

from datasets.dataset import TrafficDataset, get_data_loader, CLASS_MAP
print("Setup complete. Imports successful!")

## 1. Initialize Dataset

Define directories and create an instance of `TrafficDataset`.

In [ ]:
img_dir = "../data/UA-DETRAC/DETRAC-Images/DETRAC-Images"
anno_dir = "../data/UA-DETRAC/DETRAC-Train-Annotations-XML/DETRAC-Train-Annotations-XML"

print(f"Loading dataset from:\nImages: {img_dir}\nAnnotations: {anno_dir}")
dataset = TrafficDataset(img_dir=img_dir, anno_dir=anno_dir, img_size=(640, 640))
print(f"Successfully loaded dataset! Total images: {len(dataset)}")

## 2. Dataset Statistics & Class Verification

### Verification of Dataset Classes
There are two main configurations of classes for the UA-DETRAC dataset depending on the version used:
1. **Official UA-DETRAC dataset**: Contains exactly **4 classes**: `car`, `van` (transporter), `bus`, and `others` (which includes trucks, trailers, etc.).
2. **Re-annotated UA-DETRAC-REANNOTATED-CLASSES**: A custom version that expands categories into **9 classes**: `bicycle`, `motorcycle`, `car`, `transporter (van)`, `bus`, `truck (others)`, `trailer` (no instances), `unknown`, and `mask`.

Let\'s run a full scan of all XML files on disk in the project to verify what classes actually exist in this specific version.

In [ ]:
import xml.etree.ElementTree as ET
from collections import Counter

anno_paths = [
    "../data/UA-DETRAC/DETRAC-Train-Annotations-XML/DETRAC-Train-Annotations-XML",
    "../data/UA-DETRAC/DETRAC-Test-Annotations-XML/DETRAC-Test-Annotations-XML"
]

all_detected_classes = Counter()

for path in anno_paths:
    if os.path.exists(path):
        xml_files = [f for f in os.listdir(path) if f.endswith(".xml")]
        for xml_file in xml_files:
            tree = ET.parse(os.path.join(path, xml_file))
            for attr in tree.findall(".//attribute"):
                v_type = attr.get("vehicle_type")
                if v_type:
                    all_detected_classes[v_type.lower()] += 1

print("=== EXACT CLASSES IN THE ANNOTATION XML FILES ===")
for cls_name, count in sorted(all_detected_classes.items()):
    print(f"Class \""{cls_name}\"": {count} instances")
print("===============================================")
print(f"Conclusion: This dataset has exactly {len(all_detected_classes)} classes: {list(sorted(all_detected_classes.keys()))}.")

### Scanning a subset using TrafficDataset
Now, we scan a subset of the dataset objects via `TrafficDataset` to confirm labels and bounding box stats match the mapping.

In [ ]:
class_names = {v: k for k, v in CLASS_MAP.items()}
class_counts = {name: 0 for name in class_names.values()}
boxes_per_img = []

scan_limit = min(len(dataset), 100)
print(f"Scanning first {scan_limit} images for detailed statistics...")

for i in range(scan_limit):
    _, target = dataset[i]
    boxes = target["boxes"]
    labels = target["labels"]
    
    boxes_per_img.append(len(boxes))
    for label in labels:
        cls_name = class_names.get(label.item(), "others")
        class_counts[cls_name] += 1

print("\n================ DATASET STATISTICS ================")
print(f"Class distribution in first {scan_limit} images:")
for cls_name, count in class_counts.items():
    print(f"  - {cls_name:10s}: {count}")
    
print("\nBounding Boxes per Image statistics:")
print(f"  - Min bounding boxes:   {np.min(boxes_per_img)}")
print(f"  - Max bounding boxes:   {np.max(boxes_per_img)}")
print(f"  - Mean bounding boxes:  {np.mean(boxes_per_img):.2f}")
print(f"  - Median bounding boxes:{np.median(boxes_per_img):.2f}")
print("====================================================")

## 3. Visualize Sample Images

Plot a sample image and draw its bounding boxes with corresponding labels.

In [ ]:
# Find first image containing some objects to make visualization interesting
sample_idx = 0
for i in range(len(dataset)):
    _, target = dataset[i]
    if len(target["boxes"]) > 0:
        sample_idx = i
        break

img_tensor, target = dataset[sample_idx]
file_name = target["file_name"]
boxes = target["boxes"]
labels = target["labels"]

# Convert image tensor to [H, W, C] numpy format for visualization
# Image in TrafficDataset is normalized to [0, 1] range, convert back to [0, 255]
img = (img_tensor.permute(1, 2, 0).numpy() * 255.0).astype(np.uint8)
img_draw = img.copy()
h, w, _ = img_draw.shape

colors = {
    0: (255, 0, 0),    # Car - Red
    1: (0, 255, 0),    # Van - Green
    2: (0, 0, 255),    # Bus - Blue
    3: (255, 165, 0)   # Others - Orange
}

for box, label in zip(boxes, labels):
    # Coordinates are normalized [x1, y1, x2, y2]
    x1, y1, x2, y2 = box.numpy()
    ix1, iy1 = int(x1 * w), int(y1 * h)
    ix2, iy2 = int(x2 * w), int(y2 * h)
    
    cls_id = label.item()
    cls_name = class_names.get(cls_id, "others")
    color = colors.get(cls_id, (255, 255, 255))
    
    # Draw bbox
    cv2.rectangle(img_draw, (ix1, iy1), (ix2, iy2), color, 2)
    
    # Add text label
    label_text = f"{cls_name}"
    cv2.putText(img_draw, label_text, (ix1, max(iy1 - 5, 15)), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1, cv2.LINE_AA)

plt.figure(figsize=(12, 12))
plt.imshow(img_draw)
plt.title(f"Index: {sample_idx} | File: {file_name} ({len(boxes)} boxes)")
plt.axis("off")
plt.show()

## 4. Test DataLoader

Ensure the dataset collates correctly into batches.

In [ ]:
dataloader = get_data_loader(img_dir=img_dir, anno_dir=anno_dir, batch_size=4)
images, targets = next(iter(dataloader))

print(f"DataLoader test batch size: {len(images)}")
print(f"Batch Image tensor shape: {torch.stack(images).shape}")
print("Sample target dictionary contents:")
for k, v in targets[0].items():
    if isinstance(v, torch.Tensor):
        print(f"  - {k}: shape {v.shape}, dtype {v.dtype}")
    else:
        print(f"  - {k}: {v}")